In [1]:
import csv
import os
import glob
import sys
from pathlib import Path

import numpy as np
from sklearn.model_selection import train_test_split

CUDA_BIN = Path(r"C:/Program Files/NVIDIA GPU Computing Toolkit/CUDA/v11.5/bin")
CUDNN_BIN_CANDIDATES = [
    Path.cwd() / "third_party" / "cudnn-8.9.7-cuda11" / "bin",
    Path.cwd() / "cv_hands" / "third_party" / "cudnn-8.9.7-cuda11" / "bin",
]

dll_paths = []
if CUDA_BIN.exists():
    dll_paths.append(str(CUDA_BIN))

for candidate in CUDNN_BIN_CANDIDATES:
    if candidate.exists():
        dll_paths.append(str(candidate))
        break

if dll_paths:
    os.environ["PATH"] = ";".join(dll_paths + [os.environ.get("PATH", "")])

print("Python:", sys.executable)
print("CWD:", Path.cwd())
print("DLL search paths:", dll_paths)

import tensorflow as tf

gpus = tf.config.list_physical_devices("GPU")
print("Detected GPUs:", gpus)
if gpus:
    for gpu in gpus:
        tf.config.experimental.set_memory_growth(gpu, True)

RANDOM_SEED = 42

Python: c:\Users\User\AppData\Local\Programs\Python\Python310\python.exe
CWD: c:\Users\User\Documents\GitHub\FYP-SignLanguage\cv_hands
DLL search paths: []
Detected GPUs: []


# Dataset Path

In [2]:
model_save_path = 'model/keypoint_classifier/keypoint_sequence_classifier.keras'
tflite_save_path = 'model/keypoint_classifier/keypoint_sequence_classifier.tflite'

# Parameters

In [3]:
SEQUENCE_LENGTH = 25
FEATURES_PER_FRAME = 80  # 42 hand + 10 face + 8 pose + 20 relative

# Load Dataset

In [4]:
# Load Dataset
csv_files = sorted(glob.glob('Words-Dataset/*_sequence.csv'))
X_sequences = []
y_sequences = []
for csv_file in csv_files:
    data = np.loadtxt(csv_file, delimiter=',', dtype='float32')
    X_sequences.append(data[:, 1:].reshape(-1, SEQUENCE_LENGTH, FEATURES_PER_FRAME))
    y_sequences.extend(data[:, 0])

X_dataset = np.concatenate(X_sequences, axis=0).astype('float32')
y_dataset = np.array(y_sequences, dtype='int32')

# Remove rows with NaN/Inf to prevent val_loss becoming NaN
finite_mask = np.isfinite(X_dataset).all(axis=(1, 2))
removed_non_finite = int((~finite_mask).sum())
if removed_non_finite > 0:
    print(f"Removed non-finite samples: {removed_non_finite}")
    X_dataset = X_dataset[finite_mask]
    y_dataset = y_dataset[finite_mask]

# Extra safety: replace any residual invalid values
X_dataset = np.nan_to_num(X_dataset, nan=0.0, posinf=1e3, neginf=-1e3).astype('float32')

# Convert from 1-based to 0-based indexing for TensorFlow
y_dataset -= 1

# Load labels to determine number of classes
with open('Word-Label/keypoint_sequence_classifier_label.csv', encoding='utf-8-sig') as f:
    keypoint_sequence_classifier_labels = csv.reader(f)
    keypoint_sequence_classifier_labels = [row[0] for row in keypoint_sequence_classifier_labels]
NUM_CLASSES = len(keypoint_sequence_classifier_labels)

# Remove invalid labels outside [0, NUM_CLASSES-1]
valid_label_mask = (y_dataset >= 0) & (y_dataset < NUM_CLASSES)
removed_invalid_labels = int((~valid_label_mask).sum())
if removed_invalid_labels > 0:
    print(f"Removed invalid-label samples: {removed_invalid_labels}")
    X_dataset = X_dataset[valid_label_mask]
    y_dataset = y_dataset[valid_label_mask]

X_train, X_test, y_train, y_test = train_test_split(
    X_dataset, y_dataset, train_size=0.75, random_state=RANDOM_SEED
 )

print(f"Number of classes: {NUM_CLASSES}")
print(f"Labels: {keypoint_sequence_classifier_labels}")
print(f"Dataset shape: {X_dataset.shape}")
print(f"Train/Test: {X_train.shape} / {X_test.shape}")
print(f"Finite check train/test: {np.isfinite(X_train).all()} / {np.isfinite(X_test).all()}")

Removed invalid-label samples: 1
Number of classes: 155
Labels: ['你好', '學校', '同學', '屋企人', '鐘意', '唔鐘意', '點解', '彩虹', '謝謝', '等等', '對不起', '聾人', '我', '健聽', '\u2060手語', '現在', '高級', '認識/見面', '開心', '再見', '香港', '早餐', '星期一', '護士', 'ok', '什麼', '麵', '紙巾', '有', '類別', '願望', '星期五', '好', '懲罰', '上個星期', '人', '是', '不是', '需要', '不需要', '幫忙', '醫生', '咖啡', '想', '不想', '爸爸', '媽媽', '父母', '哥哥', '弟弟', '姐姐', '妹妹', '星期日', '爺爺', '星期二', '最後', '分鐘', '兒子', '女兒', '老公', '老婆', '頭痛', '頭盔', '我們', '鋼琴', '你', '喝', '句子', '哪裡', '溫暖', '詞語', '工作', '標籤', '摩托車', '出糧', '不開心', '厲害', '劍擊', '同事', '文職', '網絡', '運動', '學習', '緊張', '茶', '腹瀉', '義工', '功課', '零', '一', '二', '三', '四', '睡覺', '六', '七', '牛奶', '九', '助聽器', '盲', '水', '肚餓', '渴', '飽', '美味', '朋友', '愛', '介紹', '傷心', '生氣', '害怕', '累', '病', '痛', '舒服', '冷', '熱', '大', '小', '遠', '近', '快', '慢', '新', '舊', '平', '貴', '美麗', '聰明', '強壯', '弱', '左', '右', '上面', '下', '前', '後', '最鍾意', '嘗試', '買', '教', '睇', '電話', '電影', '吃', '老師', '會去', '醫院', '假期', '責任', '一齊', '感覺', '帶', '失敗', '成功']
Dataset shape: (229335, 25, 80)

# Build LSTM Model

In [ ]:
if 'NUM_CLASSES' not in locals() or NUM_CLASSES == 0:
    print("No classes found. Please collect data first.")
else:
    from tensorflow.keras import layers
    model = tf.keras.models.Sequential([
        layers.Bidirectional(layers.LSTM(256, return_sequences=True), input_shape=(SEQUENCE_LENGTH, FEATURES_PER_FRAME)),
        layers.Dropout(0.3),
        layers.Bidirectional(layers.LSTM(128)),
        layers.Dropout(0.3),
        layers.Dense(128, activation='relu'),
        layers.BatchNormalization(),
        layers.Dropout(0.3),
        layers.Dense(NUM_CLASSES, activation='softmax')
    ])

    model.summary()

Model: "sequential_2"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 bidirectional_4 (Bidirectio  (None, 25, 512)          690176    
 nal)                                                            
                                                                 
 dropout_6 (Dropout)         (None, 25, 512)           0         
                                                                 
 bidirectional_5 (Bidirectio  (None, 256)              656384    
 nal)                                                            
                                                                 
 dropout_7 (Dropout)         (None, 256)               0         
                                                                 
 dense_4 (Dense)             (None, 128)               32896     
                                                                 
 batch_normalization_2 (Batc  (None, 128)             

# Compile and Train Model

In [ ]:
model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])

physical_gpus = tf.config.list_physical_devices('GPU')
logical_gpus = tf.config.list_logical_devices('GPU')
print('Physical GPUs:', physical_gpus)
print('Logical GPUs:', logical_gpus)
if logical_gpus:
    print('Training device target: GPU')
else:
    print('Training device target: CPU (GPU not detected)')

print('Validation finite check:', np.isfinite(X_test).all(), np.isfinite(y_test).all())

# Keep only ONE checkpoint file (best model)
cp_callback = tf.keras.callbacks.ModelCheckpoint(
    model_save_path,
    monitor='val_accuracy',
    mode='max',
    verbose=1,
    save_weights_only=False,
    save_best_only=True
)

es_callback = tf.keras.callbacks.EarlyStopping(
    monitor='val_accuracy',
    mode='max',
    patience=20,  # change early stop here
    restore_best_weights=True,
    verbose=1
)
nan_callback = tf.keras.callbacks.TerminateOnNaN()

history = model.fit(
    X_train, y_train,
    epochs=2000,
    batch_size=32,
    validation_data=(X_test, y_test),
    callbacks=[cp_callback, es_callback, nan_callback]
 )

Physical GPUs: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]
Logical GPUs: [LogicalDevice(name='/device:GPU:0', device_type='GPU')]
Training device target: GPU
Validation finite check: True True
Epoch 1/2000
5376/5376 [==============================] - ETA: 0s - loss: 0.4679 - accuracy: 0.8845
Epoch 1: val_accuracy improved from -inf to 0.94082, saving model to model/keypoint_classifier\keypoint_sequence_classifier.keras
5376/5376 [==============================] - 68s 12ms/step - loss: 0.4679 - accuracy: 0.8845 - val_loss: 0.2322 - val_accuracy: 0.9408
Epoch 2/2000
5374/5376 [============================>.] - ETA: 0s - loss: 0.0674 - accuracy: 0.9812
Epoch 2: val_accuracy improved from 0.94082 to 0.98988, saving model to model/keypoint_classifier\keypoint_sequence_classifier.keras
5376/5376 [==============================] - 76s 14ms/step - loss: 0.0674 - accuracy: 0.9812 - val_loss: 0.0337 - val_accuracy: 0.9899
Epoch 3/2000
5373/5376 [===========================

KeyboardInterrupt: 

# Save a full SavedModel for OpenVINO conversion
Add a SavedModel export to ensure weights and graph are fully serialized.

In [ ]:
saved_model_dir = 'model/keypoint_classifier/keypoint_sequence_classifier_savedmodel'
if 'model' in locals():
    os.makedirs(saved_model_dir, exist_ok=True)
    model.save(saved_model_dir)
    print(f"SavedModel exported to: {saved_model_dir}")
else:
    print("Model not found. Train the model first.")

INFO:tensorflow:Assets written to: model/keypoint_classifier/keypoint_sequence_classifier_savedmodel\assets


INFO:tensorflow:Assets written to: model/keypoint_classifier/keypoint_sequence_classifier_savedmodel\assets


SavedModel exported to: model/keypoint_classifier/keypoint_sequence_classifier_savedmodel


# Evaluate Model

In [ ]:
val_loss, val_acc = model.evaluate(X_test, y_test)
print(f'Validation Loss: {val_loss}, Validation Accuracy: {val_acc}')

1792/1792 [==============================] - 8s 4ms/step - loss: 0.0016 - accuracy: 0.9996
Validation Loss: 0.0015866081230342388, Validation Accuracy: 0.9995813965797424


# Convert to TFLite

In [ ]:
model.save(model_save_path)

converter = tf.lite.TFLiteConverter.from_keras_model(model)
converter.optimizations = [tf.lite.Optimize.DEFAULT]
converter.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS, tf.lite.OpsSet.SELECT_TF_OPS]
converter._experimental_lower_tensor_list_ops = False
tflite_model = converter.convert()

with open(tflite_save_path, 'wb') as f:
    f.write(tflite_model)

print("TFLite model saved.")

INFO:tensorflow:Assets written to: C:\Users\240065951\AppData\Local\Temp\tmphouts7ta\assets


INFO:tensorflow:Assets written to: C:\Users\240065951\AppData\Local\Temp\tmphouts7ta\assets


TFLite model saved.


# Test Inference

In [ ]:
# Test Inference
interpreter = tf.lite.Interpreter(model_path=tflite_save_path)

# Add Flex delegate for TensorFlow ops
from tensorflow.lite.python.interpreter import load_delegate
try:
    delegate = load_delegate('libtensorflowlite_flex.so')
    interpreter = tf.lite.Interpreter(model_path=tflite_save_path, experimental_delegates=[delegate])
except:
    print("Flex delegate not available, using standard interpreter")

interpreter.allocate_tensors()

input_details = interpreter.get_input_details()
output_details = interpreter.get_output_details()

interpreter.set_tensor(input_details[0]['index'], np.array([X_test[0]], dtype=np.float32))
interpreter.invoke()
result = interpreter.get_tensor(output_details[0]['index'])

print("Predicted:", np.argmax(result))
print("Actual:", y_test[0])

Exception ignored in: <function Delegate.__del__ at 0x00000244992F49D0>
Traceback (most recent call last):
  File "c:\Users\240065951\Documents\GitHub\FYP-SignLanguage\cv_hands\.venv-train310\lib\site-packages\tensorflow\lite\python\interpreter.py", line 117, in __del__
    if self._library is not None:
AttributeError: 'Delegate' object has no attribute '_library'


Flex delegate not available, using standard interpreter
Predicted: 102
Actual: 102


In [ ]:
import sys
import subprocess

try:
    import matplotlib.pyplot as plt
except ModuleNotFoundError:
    print('matplotlib not found, installing...')
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'matplotlib'])
    import matplotlib.pyplot as plt

if 'history' not in locals():
    print('No history found. Run training first (Cell 11).')
else:
    hist = history.history
    epochs = range(1, len(hist.get('accuracy', [])) + 1)

    plt.figure(figsize=(12, 5))

    plt.subplot(1, 2, 1)
    plt.plot(epochs, hist.get('accuracy', []), label='train_accuracy')
    plt.plot(epochs, hist.get('val_accuracy', []), label='val_accuracy')
    plt.xlabel('Epoch')
    plt.ylabel('Accuracy')
    plt.title('Training vs Validation Accuracy')
    plt.legend()
    plt.grid(alpha=0.3)

    plt.subplot(1, 2, 2)
    plt.plot(epochs, hist.get('loss', []), label='train_loss')
    plt.plot(epochs, hist.get('val_loss', []), label='val_loss')
    plt.xlabel('Epoch')
    plt.ylabel('Loss')
    plt.title('Training vs Validation Loss')
    plt.legend()
    plt.grid(alpha=0.3)

    plt.tight_layout()
    plt.show()

No history found. Run training first (Cell 11).
